# Notebook 04 — Inference Only (v2)

This notebook **does not build indices**. It only loads the measurement *bundle* produced by Notebook 05 and runs inference (effect sizes + uncertainty + bootstrap sign stability + arc tests).

In [17]:
# ==============================
# SECTION 0 — Setup & bundle discovery
# ==============================
from __future__ import annotations

from pathlib import Path
import os
import re
import numpy as np
import pandas as pd

def normalize_id(x) -> str:
    s = str(x).strip()
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]
    return s

def find_results_dir(search_start: Path, max_depth: int = 4) -> Path | None:
    """
    Recursively look upward and in subdirs for a "results" folder.
    """
    current = search_start.resolve()
    for _ in range(max_depth + 1):
        results = current / "results"
        if results.is_dir():
            return results
        # Also look in likely subfolders (e.g., under 04_hypothesis_testing)
        for sub in current.iterdir():
            if sub.is_dir() and sub.name == "results":
                return sub
        if current.parent == current:
            break
        current = current.parent
    return None

PROJECT_ROOT = Path(os.environ.get("ROMANCE_ROOT", str(Path.cwd()))).resolve()
RESULTS_DIR = PROJECT_ROOT / "results"
DATA_DIR = PROJECT_ROOT / "data"

# Try to find the "results" directory if not present
if not RESULTS_DIR.exists():
    print(f"WARNING: RESULTS_DIR not found at expected path: {RESULTS_DIR}")
    found_results = find_results_dir(PROJECT_ROOT)
    if found_results is not None:
        print(f"Discovered results directory at: {found_results}")
        RESULTS_DIR = found_results
        # Update PROJECT_ROOT based on discovered results directory
        PROJECT_ROOT = RESULTS_DIR.parent
        DATA_DIR = PROJECT_ROOT / "data"
    else:
        # Try from this script's directory (might be running from another ipynb location)
        here = Path(__file__).parent if "__file__" in globals() else Path.cwd()
        found_results = find_results_dir(here)
        if found_results is not None:
            print(f"Discovered results directory at: {found_results}")
            RESULTS_DIR = found_results
            # Update PROJECT_ROOT based on discovered results directory
            PROJECT_ROOT = RESULTS_DIR.parent
            DATA_DIR = PROJECT_ROOT / "data"
        else:
            raise FileNotFoundError(
                f"Could not locate a 'results' directory from either {PROJECT_ROOT} or {here}. "
                "Please set ROMANCE_ROOT or create a results directory."
            )

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RESULTS_DIR: {RESULTS_DIR}")
print(f"DATA_DIR: {DATA_DIR}")

def find_latest_bundle(search_root: Path) -> Path:
    """
    Find the most recently modified directory containing core_variables.csv.
    """
    candidates = []
    if not search_root.exists():
        raise FileNotFoundError(f"Results dir not found: {search_root}")
    for p in search_root.rglob("core_variables.csv"):
        candidates.append(p.parent)
    if not candidates:
        raise FileNotFoundError("Could not find any measurement bundle (core_variables.csv) under results/. Run Notebook 05 export bundle first.")
    candidates = sorted(candidates, key=lambda d: d.stat().st_mtime, reverse=True)
    return candidates[0]

bundle_dir_env = os.environ.get("MEASUREMENT_BUNDLE_DIR", "").strip()
if bundle_dir_env:
    BUNDLE_DIR = Path(bundle_dir_env).expanduser()
else:
    BUNDLE_DIR = find_latest_bundle(RESULTS_DIR)

print(f"Using BUNDLE_DIR: {BUNDLE_DIR}")


Discovered results directory at: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results
PROJECT_ROOT: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor
RESULTS_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results
DATA_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/data
Using BUNDLE_DIR: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/measurement_v5/bundle


In [18]:
# ==============================
# SECTION 1 — Load bundle tables
# ==============================
def read_csv(name: str) -> pd.DataFrame:
    path = BUNDLE_DIR / name
    if not path.exists():
        raise FileNotFoundError(f"Missing bundle file: {path}")
    return pd.read_csv(path)

composite_registry = read_csv("composite_registry.csv")
composite_diagnostics = read_csv("composite_diagnostics.csv")
pipeline_audit = read_csv("pipeline_audit.csv")
core_variables = read_csv("core_variables.csv")

book_indices_z = read_csv("book_indices_z.csv")
book_indices_raw = read_csv("book_indices_raw.csv")

# Optional bundle files
arc_contrasts_sum = None
pca_scores_long = None
prob_contract_book = None
prob_contract_segment = None

for opt in ["arc_contrasts_sum.csv", "composite_pca_scores_long.csv", "prob_contract_book.csv", "prob_contract_segment.csv", "segment_indices_raw.csv", "segment_indices_z.csv"]:
    p = BUNDLE_DIR / opt
    if p.exists():
        if opt == "arc_contrasts_sum.csv":
            arc_contrasts_sum = pd.read_csv(p)
        elif opt == "composite_pca_scores_long.csv":
            pca_scores_long = pd.read_csv(p)
        elif opt == "prob_contract_book.csv":
            prob_contract_book = pd.read_csv(p)
        elif opt == "prob_contract_segment.csv":
            prob_contract_segment = pd.read_csv(p)

print("Loaded bundle tables:")
print("  composite_registry:", composite_registry.shape)
print("  composite_diagnostics:", composite_diagnostics.shape)
print("  pipeline_audit:", pipeline_audit.shape)
print("  core_variables:", core_variables.shape)
print("  book_indices_z:", book_indices_z.shape)
print("  book_indices_raw:", book_indices_raw.shape)
print("  arc_contrasts_sum:", None if arc_contrasts_sum is None else arc_contrasts_sum.shape)
print("  pca_scores_long:", None if pca_scores_long is None else pca_scores_long.shape)

# Contract checks (these should be clean if topic probs were normalized upstream)
if prob_contract_book is not None:
    s = prob_contract_book["sum_prob"]
    print("\nBook prob sum contract (from bundle): min/median/max =", float(s.min()), float(s.median()), float(s.max()))
if prob_contract_segment is not None:
    s = prob_contract_segment["sum_prob"]
    print("Segment prob sum contract (from bundle): min/median/max =", float(s.min()), float(s.median()), float(s.max()))


Loaded bundle tables:
  composite_registry: (26, 30)
  composite_diagnostics: (26, 9)
  pipeline_audit: (26, 17)
  core_variables: (17, 9)
  book_indices_z: (92, 40)
  book_indices_raw: (92, 40)
  arc_contrasts_sum: (92, 53)
  pca_scores_long: (2392, 4)

Book prob sum contract (from bundle): min/median/max = 1.0 1.0 1.0000000000000002
Segment prob sum contract (from bundle): min/median/max = 0.9999999999999998 1.0 1.0000000000000002


In [19]:
# ==============================
# SECTION 1.1 — Load metadata (outcomes) from sentence_df (source of truth)
# ==============================
# We keep outcomes separate from measurement to avoid leakage.
SENTENCE_DF_PATH = Path(os.environ.get("SENTENCE_DF_PATH", DATA_DIR / "processed" / "sentence_df_with_topics.parquet"))
if not SENTENCE_DF_PATH.exists():
    raise FileNotFoundError(f"Missing sentence_df: {SENTENCE_DF_PATH}")

sent = pd.read_parquet(SENTENCE_DF_PATH)

# Required columns for outcomes
required = ["book_id", "rating_mean", "rating_count"]
missing = [c for c in required if c not in sent.columns]
if missing:
    raise ValueError(f"sentence_df missing required columns: {missing}. Available: {list(sent.columns)}")

sent = sent.copy()
sent["book_id"] = sent["book_id"].map(normalize_id)

# collapse to book-level metadata (take first non-null)
meta_cols = ["rating_mean", "rating_count", "rating_class", "Author", "Book Title"]
meta_cols = [c for c in meta_cols if c in sent.columns]

meta = (sent.sort_values(["book_id"])
            .groupby("book_id", as_index=False)[meta_cols]
            .first())

# Add log popularity proxy
meta["log_rating_count"] = np.log1p(meta["rating_count"].astype(float))

print("meta:", meta.shape)
print(meta.head())


meta: (92, 7)
     book_id  rating_mean  rating_count rating_class           Author  \
0  104659050         4.06         39826          mid  Catharina_Maura   
1   11266880         3.97         28526          mid         Ann_Cole   
2  123257687         3.89        103954          bad          LJ_Shen   
3  123446478         3.95         28772          mid       Rose_Shain   
4  127305713         4.25         53817         good          LJ_Shen   

                  Book Title  log_rating_count  
0      The Unwanted Marriage         10.592300  
1               I Choose You         10.258606  
2                The Villain         11.551713  
3  Between Love and Loathing         10.267193  
4                 The Hunter         10.893363  


In [20]:
# ==============================
# SECTION 1.2 — Build modeling table (indices + outcomes)
# ==============================
# Normalize ids
for df in [book_indices_z, book_indices_raw]:
    df["book_id"] = df["book_id"].map(normalize_id)

df = book_indices_z.merge(meta, on="book_id", how="inner")
print("Merged df:", df.shape)

# Add PCA scores as alternative predictors for multidimensional composites
if pca_scores_long is not None:
    pca_scores_long["book_id"] = pca_scores_long["book_id"].map(normalize_id)
    wide_pc1 = pca_scores_long.pivot_table(index="book_id", columns="composite_key", values="pc1", aggfunc="mean")
    wide_pc1.columns = [f"{c}__pc1" for c in wide_pc1.columns]
    wide_pc1 = wide_pc1.reset_index()
    df = df.merge(wide_pc1, on="book_id", how="left")
    print("Added PC1 predictors:", wide_pc1.shape)

# Add arc contrasts (end-begin and mid-begin) if available
if arc_contrasts_sum is not None:
    arc_contrasts_sum["book_id"] = arc_contrasts_sum["book_id"].map(normalize_id)
    df = df.merge(arc_contrasts_sum, on="book_id", how="left")
    print("Added arc contrasts:", arc_contrasts_sum.shape)

# Final checks
assert df["book_id"].nunique() == meta["book_id"].nunique(), "Mismatch after merge; investigate missing books."
print("Final modeling df columns:", len(df.columns))


Merged df: (92, 46)
Added PC1 predictors: (92, 27)
Added arc contrasts: (92, 53)
Final modeling df columns: 124


## SECTION 2 — Define predictor sets (CORE gating + recommended scores)

In [21]:
# CORE composites and recommended score choice
core = composite_registry.loc[composite_registry["status_core"]=="CORE", ["composite_key","recommended_score","dimensionality"]].copy()

# Map each composite to the actual column name in df:
# - recommended_score == "sum"  -> use composite_key column (already z-scored)
# - recommended_score == "pc1"  -> use f"{key}__pc1"
# - recommended_score == "atomic_sum" -> treat as exploratory (usually excluded)
def predictor_col(row) -> str:
    k = row["composite_key"]
    rs = str(row.get("recommended_score","sum"))
    if rs == "pc1":
        return f"{k}__pc1"
    return k

core["predictor_col"] = core.apply(predictor_col, axis=1)

# Exclude atomic measures from main analysis
core_main = core.loc[core["recommended_score"]!="atomic_sum"].copy()

# Keep only predictors that exist
core_main = core_main.loc[core_main["predictor_col"].isin(df.columns)].copy()

print("CORE predictors available:", core_main.shape[0])
print(core_main.head(10))

CORE_PREDICTORS = core_main["predictor_col"].tolist()


CORE predictors available: 17
              composite_key recommended_score    dimensionality  \
0         R2_alpha_guarding               sum    UNIDIMENSIONAL   
1                  Q_repair               sum    UNIDIMENSIONAL   
2  M_health_recovery_growth               sum    UNIDIMENSIONAL   
3         I_humor_lightness               sum    UNIDIMENSIONAL   
4        Q_miscommunication               sum    UNIDIMENSIONAL   
5       A2_emotional_safety               pc1  MULTIDIMENSIONAL   
6        L_vices_addictions               sum    UNIDIMENSIONAL   
7  K_professional_intrusion               sum    UNIDIMENSIONAL   
8      C_explicit_eroticism               sum    UNIDIMENSIONAL   
9           S_scene_anchors               sum    UNIDIMENSIONAL   

              predictor_col  
0         R2_alpha_guarding  
1                  Q_repair  
2  M_health_recovery_growth  
3         I_humor_lightness  
4        Q_miscommunication  
5  A2_emotional_safety__pc1  
6        L_vices_addic

## SECTION 3 — Effect sizes + bootstrap sign stability (pilot-appropriate inference)

In [22]:
import numpy as np
import pandas as pd

def zscore(x: pd.Series) -> pd.Series:
    x = pd.to_numeric(x, errors="coerce")
    return (x - x.mean()) / (x.std(ddof=0) + 1e-12)

def ols_beta(X: np.ndarray, y: np.ndarray) -> float:
    # returns coefficient on first column of X (assumes X includes intercept + predictor + optional controls)
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    return float(b[1])

def bootstrap_beta(df_: pd.DataFrame, y_col: str, x_col: str, controls: list[str] = None, n_boot: int = 1000, seed: int = 11):
    controls = controls or []
    rng = np.random.default_rng(seed)
    rows = []
    n = len(df_)
    idx = np.arange(n)
    for _ in range(n_boot):
        samp = rng.choice(idx, size=n, replace=True)
        d = df_.iloc[samp]
        y = zscore(d[y_col]).to_numpy()
        x = zscore(d[x_col]).to_numpy()
        X = [np.ones_like(x), x]
        for c in controls:
            X.append(zscore(d[c]).to_numpy())
        X = np.column_stack(X)
        rows.append(ols_beta(X, y))
    betas = np.array(rows, dtype=float)
    beta_hat = float(betas.mean())
    ci_low, ci_high = float(np.quantile(betas, 0.025)), float(np.quantile(betas, 0.975))
    p_pos = float((betas > 0).mean())
    return beta_hat, ci_low, ci_high, p_pos

# Choose outcomes (continuous primary)
OUTCOMES = ["rating_mean", "log_rating_count"]
OUTCOMES = [c for c in OUTCOMES if c in df.columns]

# Controls: keep minimal for small N. Use none by default; optionally add log_rating_count when outcome is rating_mean.
def controls_for(outcome: str) -> list[str]:
    if outcome == "rating_mean" and "log_rating_count" in df.columns:
        return ["log_rating_count"]
    return []

results = []
for outcome in OUTCOMES:
    ctrls = controls_for(outcome)
    for x in CORE_PREDICTORS:
        if x not in df.columns:
            continue
        # drop NA rows
        sub = df[[outcome, x] + ctrls].dropna()
        if len(sub) < 40:
            continue
        beta_hat, ci_low, ci_high, p_pos = bootstrap_beta(sub, outcome, x, controls=ctrls, n_boot=800, seed=11)
        results.append({
            "outcome": outcome,
            "predictor": x,
            "beta_std": beta_hat,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p(beta>0)": p_pos,
            "n": len(sub),
            "controls": ",".join(ctrls) if ctrls else ""
        })

res_df = pd.DataFrame(results).sort_values(["outcome", "p(beta>0)"], ascending=[True, False])
print("Top directional effects (by P(beta>0)):")
display(res_df.head(15))


Top directional effects (by P(beta>0)):


,outcome,predictor,beta_std,ci_low,ci_high,p(beta>0),n,controls
17,log_rating_count,R2_alpha_guarding,0.435101,0.225990,0.596624,1.00000,92,
29,log_rating_count,D_power_wealth_luxury__pc1,0.370879,0.195460,0.545563,1.00000,92,
22,log_rating_count,A2_emotional_safety__pc1,0.323374,0.126439,0.503229,0.99750,92,
18,log_rating_count,Q_repair,0.230164,0.020573,0.405688,0.98500,92,
27,log_rating_count,J_social_support_kin,0.191182,-0.043287,0.426160,0.94875,92,
21,log_rating_count,Q_miscommunication,0.110414,-0.214488,0.440677,0.72000,92,
20,log_rating_count,I_humor_lightness,0.036558,-0.199858,0.229649,0.66250,92,
24,log_rating_count,K_professional_intrusion,0.044871,-0.166195,0.252707,0.64375,92,
19,log_rating_count,M_health_recovery_growth,0.023501,-0.233905,0.204951,0.63625,92,
32,log_rating_count,R1_protective_caretaking,0.023794,-0.259109,0.283841,0.57625,92,


## SECTION 4 — Arc tests using exported deltas (end−begin, middle−begin)

This replaces any END-variant creation logic. We test arcs explicitly.

In [23]:
# Identify arc predictors
arc_cols = [c for c in df.columns if c.endswith("__end_minus_begin") or c.endswith("__middle_minus_begin")]
print("Arc predictors:", len(arc_cols))

# Example: test arcs as predictors of rating_mean (pilot-style)
arc_results = []
if "rating_mean" in df.columns and arc_cols:
    for x in arc_cols:
        sub = df[["rating_mean", x, "log_rating_count"]].dropna() if "log_rating_count" in df.columns else df[["rating_mean", x]].dropna()
        if len(sub) < 40:
            continue
        ctrls = ["log_rating_count"] if "log_rating_count" in sub.columns else []
        beta_hat, ci_low, ci_high, p_pos = bootstrap_beta(sub, "rating_mean", x, controls=ctrls, n_boot=800, seed=13)
        arc_results.append({
            "predictor": x,
            "beta_std": beta_hat,
            "ci_low": ci_low,
            "ci_high": ci_high,
            "p(beta>0)": p_pos,
            "n": len(sub)
        })

arc_df = pd.DataFrame(arc_results).sort_values("p(beta>0)", ascending=False)
display(arc_df.head(15))


Arc predictors: 52


,predictor,beta_std,ci_low,ci_high,p(beta>0),n
18,F2_anger_frustration__end_minus_begin,0.238412,0.075336,0.412193,0.99500,92
20,F3_anxiety_worry__end_minus_begin,0.189772,0.015351,0.363993,0.98125,92
49,R_jealousy_possessiveness__middle_minus_begin,0.120933,-0.000285,0.261007,0.97250,92
38,O_aesthetics_appearance__end_minus_begin,0.149828,-0.035484,0.315657,0.95000,92
35,K_professional_intrusion__middle_minus_begin,0.182151,-0.130331,0.418555,0.90500,92
47,R2_alpha_guarding__middle_minus_begin,0.088970,-0.042562,0.238130,0.89250,92
3,A2_emotional_safety__middle_minus_begin,0.134413,-0.099997,0.347082,0.87750,92
39,O_aesthetics_appearance__middle_minus_begin,0.135758,-0.109824,0.351356,0.86875,92
12,D_power_wealth_luxury__end_minus_begin,0.091592,-0.073411,0.247448,0.86750,92
34,K_professional_intrusion__end_minus_begin,0.152356,-0.095099,0.388257,0.85750,92


## SECTION 5 — Predictive usefulness (cross-validated)

Pilot-friendly: compare metadata-only vs +CORE indices.

In [24]:
from sklearn.model_selection import KFold
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score

def cv_r2(df_: pd.DataFrame, y_col: str, X_cols: list[str], k: int = 5, seed: int = 11) -> float:
    d = df_[[y_col] + X_cols].dropna()
    if len(d) < 50:
        return np.nan
    y = d[y_col].to_numpy(dtype=float)
    X = d[X_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy(dtype=float)
    # standardize X columns
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    preds = np.zeros_like(y)
    for train, test in kf.split(X):
        model = RidgeCV(alphas=np.logspace(-3, 3, 25))
        model.fit(X[train], y[train])
        preds[test] = model.predict(X[test])
    return float(r2_score(y, preds))

# Model sets
meta_features = [c for c in ["log_rating_count"] if c in df.columns]
core_features = CORE_PREDICTORS

pred_rows = []
if "rating_mean" in df.columns:
    pred_rows.append({"outcome":"rating_mean","model":"metadata_only","cv_r2":cv_r2(df,"rating_mean",meta_features)})
    pred_rows.append({"outcome":"rating_mean","model":"metadata+core","cv_r2":cv_r2(df,"rating_mean",meta_features+core_features)})

if "log_rating_count" in df.columns:
    # popularity outcome: use only core features (no need to include itself)
    pred_rows.append({"outcome":"log_rating_count","model":"core_only","cv_r2":cv_r2(df,"log_rating_count",core_features)})

pred_df = pd.DataFrame(pred_rows)
display(pred_df)


,outcome,model,cv_r2
0,rating_mean,metadata_only,0.124947
1,rating_mean,metadata+core,0.049565
2,log_rating_count,core_only,0.033632
